# MLP vs CNN
## Architecture, Efficiency, and Inductive Bias

| | |
|---|---|
| **Auther** | Kamva Poswa |
| **Focus** | Artificial Intelligence |

---

### Introduction

I am are investigating two foundational deep learning architectures; the Multi-Layer Perceptron (MLP) and the Convolutional Neural Network (CNN), through two lenses:

- **Part A (Complexity Ladder):** Train a fixed MLP and CNN on MNIST and CIFAR-10, we observe how performance diverges as task complexity increases.
- **Part B (Parameter Budget):** Under a hard budget parameters, we determine how much architecture design matters compared to raw parameter count.

**Input Standardisation Strategy:** All images are resized to 32×32 using a dataloader transform. This strategy is applied consistently across both datasets.

## Setting up libraries; Imports, Seeds, and Device

In [ ]:


import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import time
import itertools

# Reproducibility — all random seeds fixed
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Data Loading

Both datasets are loaded using `torchvision.datasets`. MNIST is resized to 32×32 to match CIFAR-10 dimensions, fulfilling the input standardisation requirement.

In [ ]:
# Standardise all images to 32x32
IMG_SIZE   = 32
BATCH_SIZE = 128

# MNIST: greyscale, resized to 32x32. Normalised with MNIST channel mean and std.
mnist_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

# CIFAR-10: already 32x32 RGB. Normalised per channel with CIFAR-10 statistics.
cifar_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2470, 0.2435, 0.2616))
])

# Download and load datasets
mnist_train = torchvision.datasets.MNIST(root='./data', train=True,  download=True, transform=mnist_transform)
mnist_test  = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=mnist_transform)
cifar_train = torchvision.datasets.CIFAR10(root='./data', train=True,  download=True, transform=cifar_transform)
cifar_test  = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=cifar_transform)

# DataLoaders — shuffle training data each epoch
mnist_train_loader = DataLoader(mnist_train, batch_size=BATCH_SIZE, shuffle=True)
mnist_test_loader  = DataLoader(mnist_test,  batch_size=BATCH_SIZE, shuffle=False)
cifar_train_loader = DataLoader(cifar_train, batch_size=BATCH_SIZE, shuffle=True)
cifar_test_loader  = DataLoader(cifar_test,  batch_size=BATCH_SIZE, shuffle=False)

print(f"MNIST   — Train: {len(mnist_train):,} | Test: {len(mnist_test):,}")
print(f"CIFAR-10 — Train: {len(cifar_train):,} | Test: {len(cifar_test):,}")

# Sanity check — confirm shapes
images, labels = next(iter(mnist_train_loader))
print(f"\nMNIST  batch shape: {images.shape}")
images_c, _ = next(iter(cifar_train_loader))
print(f"CIFAR-10 batch shape: {images_c.shape}")

---
## Part A - The Complexity Ladder

The goal of Part A is to train a fixed MLP and a fixed CNN on both MNIST and CIFAR-10, observing at what point of task difficulty the performance gap between architectures opens and understanding why.

### A4.1 - Implementation

#### MLP Architecture

The MLP uses 3 hidden fully-connected layers (512 -> 256 -> 128 -> 10). The core architecture is **identical across both datasets** btu only the input size changes to reflect 1 vs 3 channels. Each hidden layer is followed by ReLU activation and Dropout(0.3) for regularisation.

In [ ]:
# A4.1: MLP Architecture

class MLP(nn.Module):
    def __init__(self, input_size, num_classes=10, dropout=0.3):
        
        # input_size  : flattened image size, 1024 for MNIST and 3072 for CIFAR-10
        # num_classes : output categories, 10 for both datasets
        # dropout     : fraction of neurons randomly disabled during training
        
        super(MLP, self).__init__()

        # Core architecture — identical across datasets, only input_size differs
        self.network = nn.Sequential(
            nn.Flatten(),                      # Flatten 2D image to 1D vector

            nn.Linear(input_size, 512),        # Hidden layer 1
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(512, 256),               # Hidden layer 2
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(256, 128),               # Hidden layer 3
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(128, num_classes)        # Output layer
        )

    def forward(self, x):
        return self.network(x)


# This part is for my own observation, and I figured it could be helpful to the marker too: count trainable parameters
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# Instantiate models
mlp_mnist = MLP(input_size=1*32*32).to(device)   # 1 channel × 32 × 32 = 1024
mlp_cifar = MLP(input_size=3*32*32).to(device)   # 3 channels × 32 × 32 = 3072
print(f"MLP (MNIST input)  - parameters: {count_params(mlp_mnist):,}")
print(f"MLP (CIFAR input)  - parameters: {count_params(mlp_cifar):,}")

#### CNN Architecture

The CNN uses 3 convolutional layers (Conv → BatchNorm → ReLU → MaxPool) followed by a 2-layer fully-connected head. The core architecture is **identical across both datasets**, only 'in_channels' changes (1 for MNIST, 3 for CIFAR-10). Each MaxPool halves the spatial dimensions: 32×32 → 16×16 → 8×8 → 4×4.

In [ ]:
# A4.1: CNN Architecture

class CNN(nn.Module):
    def __init__(self, in_channels, num_classes=10, dropout=0.3):
        
        # in_channels : 1 for MNIST (greyscale), 3 for CIFAR-10 (RGB)
        # num_classes : output categories, 10 for both datasets
        # dropout     : dropout probability for fully-connected layers
        
        super(CNN, self).__init__()

        # Core convolutional blocks — identical across datasets
        self.conv_blocks = nn.Sequential(
            # Layer 1: [B, in_ch, 32, 32] → [B, 32, 16, 16]
            nn.Conv2d(in_channels, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            # Layer 2: outputs [B, 64, 8, 8]
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            # Layer 3: output [B, 128, 4, 4]
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
        )

        # Fully-connected head: 128 × 4 × 4 = 2048 
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(2048, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.conv_blocks(x)
        x = self.classifier(x)
        return x


# Instantiating, in_channels is the only difference
cnn_mnist = CNN(in_channels=1).to(device)
cnn_cifar = CNN(in_channels=3).to(device)

# Note: CNN parameter counts are nearly identical regardless of input channels.
# This illustrates weight sharing — a key advantage over the MLP.

print(f"CNN (MNIST input)  - parameters: {count_params(cnn_mnist):,}")
print(f"CNN (CIFAR input)  - parameters: {count_params(cnn_cifar):,}")


#### Training Loop

I used single shared training loop for all experiments. It records train and validation loss/accuracy per epoch, switches between 'model.train()' and 'model.eval()' appropriately, and uses 'torch.no_grad()' during validation to save memory.

In [ ]:
# A4.1: Training Loop

def train_model(model, train_loader, test_loader, num_epochs=30, lr=0.001):
    
    # Training a model and returns a history dict of loss/accuracy per epoch.
    # Used for all Part A and Part B experiments.
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(num_epochs):

        # Training phase
        model.train()   # enables dropout and batchnorm updates
        running_loss, correct, total = 0.0, 0, 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)              # Forward pass
            loss = criterion(outputs, labels)    # Compute loss

            optimizer.zero_grad()                # Clear previous gradients
            loss.backward()                      # Backpropagation
            optimizer.step()                     # Update weights

            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total   += labels.size(0)
            correct += predicted.eq(labels).sum().item()

        train_loss = running_loss / len(train_loader)
        train_acc  = 100. * correct / total

        # Validation phase
        model.eval()    # disables dropout
        val_loss, correct, total = 0.0, 0, 0

        with torch.no_grad():   # no gradient computation needed
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                _, predicted = outputs.max(1)
                total   += labels.size(0)
                correct += predicted.eq(labels).sum().item()

        val_loss = val_loss / len(test_loader)
        val_acc  = 100. * correct / total

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        if (epoch + 1) % 5 == 0:
            print(f"Epoch [{epoch+1:>2}/{num_epochs}] "
                  f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | "
                  f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")

    return history

### Part A - Running All Experiments

All four model/dataset combinations are trained below. Hardware used: Kaggle GPU (NVIDIA Tesla T4). Seeds are fixed to ensure reproducibility.

In [ ]:
# Part A: Run All Experiments
NUM_EPOCHS = 30

print("=" * 60)
print("Training MLP on MNIST")
print("=" * 60)
start = time.time()
mlp_mnist_history = train_model(mlp_mnist, mnist_train_loader, mnist_test_loader, num_epochs=NUM_EPOCHS)
print(f"Total Time: {time.time() - start:.1f}s\n")

print("=" * 60)
print("Training MLP on CIFAR-10")
print("=" * 60)
start = time.time()
mlp_cifar_history = train_model(mlp_cifar, cifar_train_loader, cifar_test_loader, num_epochs=NUM_EPOCHS)
print(f"Total Time: {time.time() - start:.1f}s\n")

print("=" * 60)
print("Training CNN on MNIST")
print("=" * 60)
start = time.time()
cnn_mnist_history = train_model(cnn_mnist, mnist_train_loader, mnist_test_loader, num_epochs=NUM_EPOCHS)
print(f"Total Time: {time.time() - start:.1f}s\n")

print("=" * 60)
print("Training CNN on CIFAR-10")
print("=" * 60)
start = time.time()
cnn_cifar_history = train_model(cnn_cifar, cifar_train_loader, cifar_test_loader, num_epochs=NUM_EPOCHS)
print(f"Total Time: {time.time() - start:.1f}s\n")

### A4.2 - Results Table

In [ ]:
# A4.2: Results Table
print("=" * 50)
print("     PART A - TEST ACCURACY RESULTS")
print("=" * 50)
print(f"{'Model':<8} {'Dataset':<12} {'Test Accuracy':>14}")
print("-" * 50)
print(f"{'MLP':<8} {'MNIST':<12} {mlp_mnist_history['val_acc'][-1]:>13.2f}%")
print(f"{'MLP':<8} {'CIFAR-10':<12} {mlp_cifar_history['val_acc'][-1]:>13.2f}%")
print(f"{'CNN':<8} {'MNIST':<12} {cnn_mnist_history['val_acc'][-1]:>13.2f}%")
print(f"{'CNN':<8} {'CIFAR-10':<12} {cnn_cifar_history['val_acc'][-1]:>13.2f}%")
print("=" * 50)
print(f"\nMLP accuracy drop (MNIST→CIFAR): ",
      f"{mlp_mnist_history['val_acc'][-1] - mlp_cifar_history['val_acc'][-1]:.2f} pp")
print(f"CNN accuracy drop (MNIST→CIFAR): ",
      f"{cnn_mnist_history['val_acc'][-1] - cnn_cifar_history['val_acc'][-1]:.2f} pp")

### A4.3 - Training Curves

Loss (top row) and accuracy (bottom row) for all four experiments. Both training and validation curves are shown to reveal overfitting and underfitting patterns.

In [ ]:
# A4.3: Training Curves
fig, axes = plt.subplots(2, 4, figsize=(22, 9))
fig.suptitle('Part A - Training Curves: MLP vs CNN on MNIST and CIFAR-10', fontsize=13, fontweight='bold')

configs = [
    (mlp_mnist_history,  'MLP - MNIST'),
    (mlp_cifar_history,  'MLP - CIFAR-10'),
    (cnn_mnist_history,  'CNN - MNIST'),
    (cnn_cifar_history,  'CNN - CIFAR-10'),
]

for i, (hist, title) in enumerate(configs):
    # Loss (top row)
    axes[0, i].plot(hist['train_loss'], label='Train Loss', linewidth=2)
    axes[0, i].plot(hist['val_loss'],   label='Val Loss',   linewidth=2, linestyle='--')
    axes[0, i].set_title(title, fontweight='bold')
    axes[0, i].set_xlabel('Epoch')
    axes[0, i].set_ylabel('Cross-Entropy Loss')
    axes[0, i].legend()
    axes[0, i].grid(True, alpha=0.3)

    # Accuracy (bottom row)
    axes[1, i].plot(hist['train_acc'], label='Train Acc', linewidth=2)
    axes[1, i].plot(hist['val_acc'],   label='Val Acc',   linewidth=2, linestyle='--')
    axes[1, i].set_title(title, fontweight='bold')
    axes[1, i].set_xlabel('Epoch')
    axes[1, i].set_ylabel('Accuracy (%)')
    axes[1, i].legend()
    axes[1, i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### A4.4 - Written Analysis

The results show that model performance is nearly identical on low-complexity image data but diverges sharply as dataset complexity increases. On MNIST, both architectures achieved very high accuracy (MLP: 98.32%, CNN: 99.28%), indicating that handwritten digit recognition is simple enough that even a fully connected network can learn the task effectively. However, the gap opens on CIFAR-10, where images are coloured, noisier, and contain more complex object structure. Here, the MLP dropped by 44.5 percentage points to 53.82%, while the CNN dropped only 20.74 points to 78.54%.

This difference is explained by **inductive bias**: CNNs are designed with assumptions that match image data. Convolutional layers exploit **spatial locality**, meaning nearby pixels are related, and detect local patterns such as edges or textures. **Weight sharing** also provides a degree of **translation invariance**, allowing learned features to remain useful when objects shift position. Most importantly, these properties are not learned from data, they are built into the architecture as prior assumptions about what image data looks like. Which contrastshow the MLP treats every pixel independently after flattening, destroying spatial structure and requiring far more parameters to learn the same relationships.

Training curves also reveal different failure modes. The CNN reached 94.64% training accuracy but only 78.54% validation accuracy, producing an **overfitting gap of about 16%**, it can model CIFAR-10 well but begins memorising training data, thus resulting to poor performance. The MLP achieved only 58.19% training accuracy and 53.82% validation accuracy, showing **underfitting**; it struggles to represent the dataset at all. Therefore, complexity exposes the importance of architecture, not just model sizing.

### A4.5 - Reflection: CIFAR-10 Property Responsible for MLP's Drop

The property most responsible for the MLP's accuracy drop on CIFAR-10 is the **spatial variability of objects**. Unlike MNIST digits, which are consistently centred against a plain background, CIFAR-10 objects appear at varying positions, scales, and orientations within the image. Because the MLP assigns independent weights to every pixel location after flattening, it cannot recognise the same feature - such as a texture or edge, when it appears in a different spatial position. This is directly visible in the training curves: the MLP only reached 58.19% training accuracy on CIFAR-10, suggesting it cannot even represent these spatially variable patterns.

---
## Part B - The Parameter Budget Challenge

Both models are designed to fall within a **hard parameter budget of 475,000–525,000 parameters**, trained only on CIFAR-10.

### B4.1 - Budget Architectures

Parameter counts are calculated as follows:
- **Linear layer:** `(input × output) + output (bias)`
- **Conv layer:** `(kernel_h × kernel_w × in_channels × out_channels) + out_channels`

In [ ]:
# B4.1: Budget MLP (around 500K parameters)

class BudgetMLP(nn.Module):
    def __init__(self, dropout=0.3):
        super(BudgetMLP, self).__init__()
        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(3072, 150),    # (3072×150)+150 = 460950
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(150, 100),     # (150×100)+100  = 15100
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(100, 50),      # (100×50)+50    = 5050
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(50, 10)        # (50×10)+10     = 510
        )

    def forward(self, x):
        return self.network(x)


# B4.1: Budget CNN (~500K parameters)

class BudgetCNN(nn.Module):
    def __init__(self, dropout=0.3):
        super(BudgetCNN, self).__init__()

        # Conv layer use very few parameters due to weight sharing
        self.conv_blocks = nn.Sequential(
            
            # Layer 1: 
            nn.Conv2d(3, 32, kernel_size=3, padding=1),   # (3×3×3×32)+32   = 896
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            # Layer 2:
            nn.Conv2d(32, 64, kernel_size=3, padding=1),  # (3×3×32×64)+64  = 18,496
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            # Layer 3:
            nn.Conv2d(64, 64, kernel_size=3, padding=1),  # (3×3×64×64)+64  = 36,928
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
        )

        # FC head: 64×4×4 = 1024 features
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(1024, 430),   # (1024×430)+430 = 440,750  - bulk of CNN params
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(430, 10)      # (430×10)+10    = 4,310
        )

    def forward(self, x):
        x = self.conv_blocks(x)
        x = self.classifier(x)
        return x


# Budget verification
budget_mlp = BudgetMLP().to(device)
budget_cnn = BudgetCNN().to(device)

mlp_params = count_params(budget_mlp)
cnn_params = count_params(budget_cnn)

print(f"Budget MLP - Total Parameters: {mlp_params:,}")
print(f"Budget CNN - Total Parameters: {cnn_params:,}")

#### Layer-by-Layer Parameter Tables

In [ ]:
# B4.1: Layer-by-layer parameter tables

def print_param_table(model, model_name):
    print(f"\n{'='*58}")
    print(f"  {model_name} - Layer-by-Layer Parameter Count")
    print(f"{'='*58}")
    print(f"{'Layer':<32} {'Shape':<18} {'Params':>8}")
    print(f"{'-'*58}")
    total = 0
    for name, param in model.named_parameters():
        if param.requires_grad:
            n = param.numel()
            total += n
            print(f"{name:<32} {str(list(param.shape)):<18} {n:>8,}")
    print(f"{'-'*58}")
    print(f"{'TOTAL':<50} {total:>8,}")
    print(f"{'='*58}")

print_param_table(budget_mlp, "Budget MLP")
print_param_table(budget_cnn, "Budget CNN")

### B4.2 - Training Results

Both models are trained for 30 epochs on CIFAR-10. I used the best configuration from the hyperparameter search (B4.3) for the final reporting.

In [ ]:
# B4.2: Hyperparameter Search Setup

learning_rates = [0.001, 0.0005]
dropout_rates  = [0.2, 0.4]
configs        = list(itertools.product(learning_rates, dropout_rates))

# CNN search
print("B4.3 - Hyperparameter Search: Budget CNN")
print(f"{'LR':<10} {'Dropout':<10} {'Best Val Acc':>13} {'Train Time':>12}")
print("-" * 50)

cnn_search_results = []
for lr, dropout in configs:
    model = BudgetCNN(dropout=dropout).to(device)
    start = time.time()
    hist  = train_model(model, cifar_train_loader, cifar_test_loader, num_epochs=30, lr=lr)
    elapsed = time.time() - start
    best_val_acc = max(hist['val_acc'])
    print(f"{lr:<10} {dropout:<10} {best_val_acc:>12.2f}% {elapsed:>10.1f}s")
    cnn_search_results.append({'lr': lr, 'dropout': dropout, 'val_acc': best_val_acc,
                                'time': elapsed, 'history': hist})

best_cnn = max(cnn_search_results, key=lambda x: x['val_acc'])
print(f"\n→ Best CNN config: LR={best_cnn['lr']} | Dropout={best_cnn['dropout']} | Acc={best_cnn['val_acc']:.2f}%")

# MLP search
print("\nB4.3 — Hyperparameter Search: Budget MLP")
print(f"{'LR':<10} {'Dropout':<10} {'Best Val Acc':>13} {'Train Time':>12}")
print("-" * 50)

mlp_search_results = []
for lr, dropout in configs:
    model = BudgetMLP(dropout=dropout).to(device)
    start = time.time()
    hist  = train_model(model, cifar_train_loader, cifar_test_loader, num_epochs=30, lr=lr)
    elapsed = time.time() - start
    best_val_acc = max(hist['val_acc'])
    print(f"{lr:<10} {dropout:<10} {best_val_acc:>12.2f}% {elapsed:>10.1f}s")
    mlp_search_results.append({'lr': lr, 'dropout': dropout, 'val_acc': best_val_acc,
                                'time': elapsed, 'history': hist})

best_mlp = max(mlp_search_results, key=lambda x: x['val_acc'])
print(f"\n→ Best MLP config: LR={best_mlp['lr']} | Dropout={best_mlp['dropout']} | Acc={best_mlp['val_acc']:.2f}%")

### B4.2 - Final Results Summary & time per Epoch

In [ ]:
# B4.2: Final results

print("=" * 62)
print("  PART B — HYPERPARAMETER SEARCH FULL RESULTS")
print("=" * 62)
print(f"{'Model':<8} {'LR':<8} {'Dropout':<10} {'Val Acc':>10} {'Time(s)':>10} {'Sec/Epoch':>10}")
print("-" * 62)
for r in cnn_search_results:
    spe = r['time'] / 30
    print(f"{'CNN':<8} {r['lr']:<8} {r['dropout']:<10} {r['val_acc']:>9.2f}% {r['time']:>9.1f}s {spe:>9.2f}s")
print()
for r in mlp_search_results:
    spe = r['time'] / 30
    print(f"{'MLP':<8} {r['lr']:<8} {r['dropout']:<10} {r['val_acc']:>9.2f}% {r['time']:>9.1f}s {spe:>9.2f}s")
print("=" * 62)
print(f"\nBest CNN → Val Acc: {best_cnn['val_acc']:.2f}% | Sec/Epoch: {best_cnn['time']/30:.2f}s")
print(f"Best MLP → Val Acc: {best_mlp['val_acc']:.2f}% | Sec/Epoch: {best_mlp['time']/30:.2f}s")
print(f"Accuracy gap (CNN - MLP): {best_cnn['val_acc'] - best_mlp['val_acc']:.2f} percentage points")

### B4.4 - Efficiency Analysis

Under a fixed parameter budget of 500K, the CNN achieved **79.72%** validation accuracy compared to the MLP's **54.89%**, a gap of **24.83%** despite near-identical parameter counts. This demonstrates that architecture design, not raw parameter count, determines model effectiveness on image data.

The difference arises from how parameters are distributed. In the Budget MLP, the majority of parameters sit in the first linear layer (3072 to 150), directly connecting raw pixel values to learned features. These connections encode no spatial assumptions; every pixel is treated independently, forcing the model to learn translation invariance from scratch, which 500K parameters are insufficient to achieve on CIFAR-10.

In the Budget CNN, the convolutional layers use only 56,000 parameters yet perform meaningful spatial feature extraction through weight sharing, the same filter detects edges or textures everywhere in the image. The remaining 445,000 parameters in the FC head then operate on compact, structured feature maps rather than raw pixels. Each parameter therefore contributes more useful information.

This is **inductive bias** in action: the CNN's architecture constrains learning in a way that matches image structure, making its parameters more efficient. Regarding computational cost, the CNN took about **16 seconds per epoch** versus the MLP's about **14 seconds**, despite similar parameter counts. This is because convolution slides filters across every spatial position of the feature map; more floating point operations than the matrix multiplications in an MLP, demonstrating that **equal parameter counts do not imply equal computational cost**.

Notably, the CNN performed best with higher dropout (0.4) while the MLP performed best with lower dropout (0.2). This is consistent with their failure modes: the CNN overfits and benefits from stronger regularisation, while the MLP underfits and is further harmed by excessive dropout.

### B4.5 - Reflection: Proposed Structural Modification to the MLP

The property identified in A4.5 was the **spatial variability of objects** in CIFAR-10, the MLP cannot recognise the same feature at different positions because it uses fully global connections with no spatial structure.

A structural modification that directly addresses this is introducing **locally-connected layers**: layers where each neuron connects only to a small spatial neighbourhood of the input (e.g. a 3×3 patch), rather than to every pixel. This enforces spatial locality, neurons only respond to nearby pixels which reduces the number of irrelevant long-range connections the model must learn to suppress.

The mechanism: by restricting receptive fields to local patches, the model is forced to build features from spatially coherent regions. This mirrors how images are actually structured. Textures and shapes are local phenomena.

However, this stops short of being a convolutional layer because the weights are **not shared** across spatial positions. Each local connection at position (i,j) has its own independent set of weights. This means the model still cannot recognise the same feature in a different location, it learns position-specific local detectors rather than universal ones. **Weight sharing is the defining property of convolution**, and its deliberate absence here is precisely what distinguishes a locally-connected layer from a conv layer.